# SASRec BPI2012 Colab Train

Colab notebook for training and comparing SASRec anchor runs on BPI 2012 COMPLETE-only data.

Current evaluation protocol:
- model selection: `full_valid_ndcg@10`
- main evaluation: `full` ranking with `@5`, `@10`, `MRR`
- supplementary evaluation: `sampled` ranking with `num_negative_samples=100`, `@5`, `@10`, `MRR`


In [2]:
import torch

print('torch version:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu name:', torch.cuda.get_device_name(0))


torch version: 2.10.0+cu128
cuda available: True
gpu name: Tesla T4


In [3]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
GITHUB_USERNAME = 'hwbuzz'

DRIVE_ROOT = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction'
REPO_DIR = '/content/time-aware-behavior-prediction'

DATA_DIR = f'{DRIVE_ROOT}/data/processed/bpi2012_complete_only'
OUTPUT_DIR = f'{DRIVE_ROOT}/outputs/sasrec_bpi2012'
NOTEBOOK_DIR = f'{DRIVE_ROOT}/notebooks'

print('DRIVE_ROOT:', DRIVE_ROOT)
print('DATA_DIR:', DATA_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)


DRIVE_ROOT: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction
DATA_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/data/processed/bpi2012_complete_only
OUTPUT_DIR: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012


In [5]:
!mkdir -p "$NOTEBOOK_DIR"
!mkdir -p "$DATA_DIR"
!mkdir -p "$OUTPUT_DIR"


In [6]:
# First clone only if needed.
%cd /content
!test -d time-aware-behavior-prediction || git clone https://github.com/$GITHUB_USERNAME/time-aware-behavior-prediction.git
%cd /content/time-aware-behavior-prediction


/content
/content/time-aware-behavior-prediction


In [7]:
# If you need the latest code from GitHub, uncomment below.
# %cd /content/time-aware-behavior-prediction
# !git pull


In [8]:
%cd /content/time-aware-behavior-prediction

skip_packages = ['pywinpty']

with open('requirements.txt', 'r', encoding='utf-8') as f:
    lines = f.readlines()

with open('requirements_colab.txt', 'w', encoding='utf-8') as f:
    for line in lines:
        pkg = line.strip().lower()
        if not any(name in pkg for name in skip_packages):
            f.write(line)

print('created requirements_colab.txt')


/content/time-aware-behavior-prediction
created requirements_colab.txt


In [9]:
!pip install -r requirements_colab.txt


In [10]:
!ls "$DATA_DIR"


events_complete_only_filtered.csv  sasrec_interactions.csv
events_encoded_time_features.csv   sasrec_interactions.txt
item_map.csv			   user_map.csv


In [11]:
%cd /content/time-aware-behavior-prediction
!mkdir -p data/processed
!cp -r "$DATA_DIR" data/processed/
!ls data/processed/bpi2012_complete_only


/content/time-aware-behavior-prediction
events_complete_only_filtered.csv  sasrec_interactions.csv
events_encoded_time_features.csv   sasrec_interactions.txt
item_map.csv			   user_map.csv


In [12]:
OUTPUT_DIR = '/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012'
INTERACTIONS_PATH = 'data/processed/bpi2012_complete_only/sasrec_interactions.txt'


## Anchor runs

These runs share the same evaluation settings:
- `--eval_protocol both`
- `--topk_list 5,10`
- `--selection_metric full_valid_ndcg@10`
- `--num_negative_samples 100`
- `--save_every_eval`


In [13]:
from pathlib import Path

output_dir = Path("/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012")
planned_run_names = [
    "anchor_v3_paper_default_seed42",
    "anchor_v3_bpi_short_context_seed42",
    "anchor_v3_bpi_mid_context_seed42",
    "anchor_v3_bpi_long_context_seed42",
    "anchor_v3_bpi_regularized_seed42",
]

for run_name in planned_run_names:
    run_dir = output_dir / run_name
    print(run_name, "EXISTS" if run_dir.exists() else "OK")


anchor_v3_paper_default_seed42 EXISTS
anchor_v3_bpi_short_context_seed42 EXISTS
anchor_v3_bpi_mid_context_seed42 EXISTS
anchor_v3_bpi_long_context_seed42 EXISTS
anchor_v3_bpi_regularized_seed42 EXISTS


In [14]:
!python src/train_sasrec.py \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012" \
  --run_name refine_v3_ml50_do025_seed42 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.25 \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --seed 42 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --selection_metric full_valid_ndcg@10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012/refine_v3_ml50_do025_seed42
epoch=1, loss=0.5066
epoch=2, loss=0.2315
epoch=3, loss=0.1700
epoch=4, loss=0.1361
epoch=5, loss=0.1203
valid [full], NDCG@5: 0.3562, HR@5: 0.5678, NDCG@10: 0.4269, HR@10: 0.7837, MRR: 0.3331
valid [sampled], NDCG@5: 0.5541, HR@5: 0.5661, NDCG@10: 0.5730, HR@10: 0.6257, MRR: 0.5705
test [full], NDCG@5: 0.2102, HR@5: 0.3892, NDCG@10: 0.2703, HR@10: 0.5764, MRR: 0.2055
test [sampled], NDCG@5: 0.2956, HR@5: 0.3522, NDCG@10: 0.3461, HR@10: 0.5068, MRR: 0.3104
saved eval checkpoint: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/o

In [15]:
!python src/train_sasrec.py \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012" \
  --run_name refine_v3_ml50_do030_seed42 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.3 \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --seed 42 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --selection_metric full_valid_ndcg@10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012/refine_v3_ml50_do030_seed42
epoch=1, loss=0.5360
epoch=2, loss=0.2475
epoch=3, loss=0.1849
epoch=4, loss=0.1493
epoch=5, loss=0.1305
valid [full], NDCG@5: 0.3155, HR@5: 0.4628, NDCG@10: 0.4192, HR@10: 0.7713, MRR: 0.3293
valid [sampled], NDCG@5: 0.5544, HR@5: 0.5666, NDCG@10: 0.5702, HR@10: 0.6159, MRR: 0.5690
test [full], NDCG@5: 0.1721, HR@5: 0.3196, NDCG@10: 0.2552, HR@10: 0.5732, MRR: 0.1881
test [sampled], NDCG@5: 0.2980, HR@5: 0.3541, NDCG@10: 0.3444, HR@10: 0.4954, MRR: 0.3101
saved eval checkpoint: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/o

In [16]:
!python src/train_sasrec.py \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012" \
  --run_name refine_v3_ml75_do030_seed42 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 75 \
  --lr 0.001 \
  --dropout_rate 0.3 \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --seed 42 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --selection_metric full_valid_ndcg@10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012/refine_v3_ml75_do030_seed42
epoch=1, loss=0.5423
epoch=2, loss=0.2566
epoch=3, loss=0.1895
epoch=4, loss=0.1550
epoch=5, loss=0.1346
valid [full], NDCG@5: 0.3953, HR@5: 0.6158, NDCG@10: 0.4410, HR@10: 0.7540, MRR: 0.3627
valid [sampled], NDCG@5: 0.5632, HR@5: 0.5782, NDCG@10: 0.5810, HR@10: 0.6336, MRR: 0.5759
test [full], NDCG@5: 0.3081, HR@5: 0.4034, NDCG@10: 0.3636, HR@10: 0.5729, MRR: 0.3320
test [sampled], NDCG@5: 0.2762, HR@5: 0.3333, NDCG@10: 0.3216, HR@10: 0.4711, MRR: 0.2890
saved eval checkpoint: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/o

In [17]:
!python src/train_sasrec.py \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012" \
  --run_name refine_v3_ml100_do030_seed42 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 100 \
  --lr 0.001 \
  --dropout_rate 0.3 \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --seed 42 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --selection_metric full_valid_ndcg@10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012/refine_v3_ml100_do030_seed42
epoch=1, loss=0.5443
epoch=2, loss=0.2581
epoch=3, loss=0.1902
epoch=4, loss=0.1567
epoch=5, loss=0.1356
valid [full], NDCG@5: 0.4215, HR@5: 0.6579, NDCG@10: 0.4627, HR@10: 0.7866, MRR: 0.3780
valid [sampled], NDCG@5: 0.5705, HR@5: 0.5853, NDCG@10: 0.5859, HR@10: 0.6330, MRR: 0.5827
test [full], NDCG@5: 0.3322, HR@5: 0.4352, NDCG@10: 0.3804, HR@10: 0.5829, MRR: 0.3464
test [sampled], NDCG@5: 0.3112, HR@5: 0.3672, NDCG@10: 0.3540, HR@10: 0.4969, MRR: 0.3234
saved eval checkpoint: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/

In [18]:
!python src/train_sasrec.py \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012" \
  --run_name refine_v3_ml50_do035_seed42 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.35 \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --seed 42 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --selection_metric full_valid_ndcg@10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012/refine_v3_ml50_do035_seed42
epoch=1, loss=0.5673
epoch=2, loss=0.2638
epoch=3, loss=0.1987
epoch=4, loss=0.1638
epoch=5, loss=0.1414
valid [full], NDCG@5: 0.3693, HR@5: 0.5758, NDCG@10: 0.4288, HR@10: 0.7550, MRR: 0.3477
valid [sampled], NDCG@5: 0.5545, HR@5: 0.5603, NDCG@10: 0.5654, HR@10: 0.5947, MRR: 0.5698
test [full], NDCG@5: 0.2279, HR@5: 0.4028, NDCG@10: 0.2804, HR@10: 0.5682, MRR: 0.2207
test [sampled], NDCG@5: 0.2626, HR@5: 0.2682, NDCG@10: 0.2957, HR@10: 0.3744, MRR: 0.2933
saved eval checkpoint: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/o

In [19]:
!python src/train_sasrec.py \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012" \
  --run_name refine_v3_ml50_do030_seed2024 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.3 \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --seed 2024 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --selection_metric full_valid_ndcg@10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012/refine_v3_ml50_do030_seed2024
epoch=1, loss=0.5987
epoch=2, loss=0.2553
epoch=3, loss=0.1894
epoch=4, loss=0.1533
epoch=5, loss=0.1350
valid [full], NDCG@5: 0.3203, HR@5: 0.4597, NDCG@10: 0.4847, HR@10: 0.9841, MRR: 0.3411
valid [sampled], NDCG@5: 0.5670, HR@5: 0.5725, NDCG@10: 0.5807, HR@10: 0.6159, MRR: 0.5882
test [full], NDCG@5: 0.2493, HR@5: 0.4880, NDCG@10: 0.3428, HR@10: 0.7839, MRR: 0.2302
test [sampled], NDCG@5: 0.6159, HR@5: 0.6206, NDCG@10: 0.6357, HR@10: 0.6841, MRR: 0.6379
saved eval checkpoint: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction

In [20]:
!python src/train_sasrec.py \
  --interactions_path data/processed/bpi2012_complete_only/sasrec_interactions.txt \
  --output_dir "/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012" \
  --run_name refine_v3_ml50_do025_seed2024 \
  --hidden_units 50 \
  --num_blocks 2 \
  --num_heads 1 \
  --maxlen 50 \
  --lr 0.001 \
  --dropout_rate 0.25 \
  --batch_size 128 \
  --num_epochs 50 \
  --eval_every 5 \
  --seed 2024 \
  --device cuda \
  --num_negative_samples 100 \
  --eval_protocol both \
  --topk_list 5,10 \
  --selection_metric full_valid_ndcg@10 \
  --save_every_eval


dataset split summary: users=13087, items=23, train_users=13087, valid_users=9658, test_users=9658, train_only_users=3429
interaction split: train=145190 (88.26%), valid=9658 (5.87%), test=9658 (5.87%), total=164506
avg_train_len=11.09, users_with_eval_targets=9658
batches_per_epoch=103
selection_metric=full_valid_ndcg@10
run_dir=/content/drive/MyDrive/ai-projects/time-aware-behavior-prediction/outputs/sasrec_bpi2012/refine_v3_ml50_do025_seed2024
epoch=1, loss=0.5681
epoch=2, loss=0.2402
epoch=3, loss=0.1763
epoch=4, loss=0.1409
epoch=5, loss=0.1248
valid [full], NDCG@5: 0.3103, HR@5: 0.4245, NDCG@10: 0.4857, HR@10: 0.9826, MRR: 0.3432
valid [sampled], NDCG@5: 0.5604, HR@5: 0.5637, NDCG@10: 0.5685, HR@10: 0.5894, MRR: 0.5784
test [full], NDCG@5: 0.2541, HR@5: 0.4921, NDCG@10: 0.4127, HR@10: 0.9937, MRR: 0.2423
test [sampled], NDCG@5: 0.6296, HR@5: 0.6420, NDCG@10: 0.6550, HR@10: 0.7218, MRR: 0.6481
saved eval checkpoint: /content/drive/MyDrive/ai-projects/time-aware-behavior-prediction

## Result lookup

Sort by `best_valid_ndcg` first. That column now means **full-ranking valid NDCG@10**, which is the model-selection metric.


In [ ]:
import pandas as pd

index_path = f"{OUTPUT_DIR}/experiment_index.csv"
df = pd.read_csv(index_path)

df = df.sort_values(
    ['best_valid_ndcg', 'best_valid_hr'],
    ascending=False
).reset_index(drop=True)

df[[
    'run_name',
    'best_epoch',
    'best_valid_ndcg',
    'best_valid_hr',
    'best_valid_mrr',
    'best_test_ndcg',
    'best_test_hr',
    'best_test_mrr',
    'checkpoint_best',
    'checkpoint_last',
]]


In [ ]:
df[[
    'run_name',
    'best_valid_full_ndcg@5',
    'best_valid_full_hr@5',
    'best_valid_full_ndcg@10',
    'best_valid_full_hr@10',
    'best_valid_full_mrr',
    'best_test_full_ndcg@5',
    'best_test_full_hr@5',
    'best_test_full_ndcg@10',
    'best_test_full_hr@10',
    'best_test_full_mrr',
]].head(10)


In [ ]:
df[[
    'run_name',
    'best_valid_sampled_ndcg@5',
    'best_valid_sampled_hr@5',
    'best_valid_sampled_ndcg@10',
    'best_valid_sampled_hr@10',
    'best_valid_sampled_mrr',
    'best_test_sampled_ndcg@5',
    'best_test_sampled_hr@5',
    'best_test_sampled_ndcg@10',
    'best_test_sampled_hr@10',
    'best_test_sampled_mrr',
]].head(10)


In [ ]:
best_row = df.iloc[0]
best_row


In [ ]:
print('best run_name:', best_row['run_name'])
print('best checkpoint:', best_row['checkpoint_best'])
print('last checkpoint:', best_row['checkpoint_last'])
print('selection metric:', best_row['best_valid_ndcg'])


In [ ]:
from pathlib import Path
import json

config = json.loads(Path(best_row['config_path']).read_text(encoding='utf-8'))
summary = json.loads(Path(best_row['metrics_summary']).read_text(encoding='utf-8'))

print('config')
display(config)
print('summary')
display(summary)


In [ ]:
history = pd.read_csv(best_row['metrics_history'])
display(history)


In [ ]:
ax = history.plot(
    x='epoch',
    y=['full_valid_ndcg@5', 'full_valid_ndcg@10', 'full_test_ndcg@5', 'full_test_ndcg@10'],
    marker='o',
    figsize=(10, 4),
    title=f"Full-ranking NDCG by epoch: {best_row['run_name']}"
)
ax.grid(True, alpha=0.3)
